# Unidad 4: Persistencia y Bases de Datos
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

En cualquier negocio digital, la **información es el activo más valioso**. Un e-commerce necesita registrar productos, stock, clientes e historial de pedidos. Una fintech requiere registrar transacciones con precisión atómica y absoluta consistencia.

Para lograr esto, necesitamos **Persistencia de Datos**. En esta unidad abordaremos la interacción de Python con bases de datos estructuradas relacionales:
1. **SQL Directo**: Consultas e interacciones directas con el motor ligero **SQLite** integrado en Python.
2. **Mapeo Objeto-Relacional (ORM)**: El estándar de la industria que traduce tablas relacionales en clases de Python y registros en objetos. Utilizaremos el ORM líder de la industria: **SQLAlchemy**.

### Objetivos de Aprendizaje:
1. Comprender la arquitectura de almacenamiento de datos relacional.
2. Conectarse a motores SQL y ejecutar operaciones CRUD mediante comandos directos de SQL en Python.
3. Comprender los conceptos de un ORM y mapear bases de datos relacionales a código estructurado.
4. Construir modelos relacionales avanzados (Relaciones 1 a N, claves foráneas) con SQLAlchemy.


## 1. Conexión SQL Directa con `sqlite3`

SQLite es un motor de base de datos relacional integrado en la biblioteca estándar de Python. Es excelente para el desarrollo ágil de prototipos porque almacena toda la base de datos en un solo archivo físico local.


In [ ]:
import sqlite3

# Conectar a la base de datos (se crea el archivo 'tienda.db' si no existe)
conexion = sqlite3.connect("tienda.db")
cursor = conexion.cursor()

# 1. Crear una tabla de productos
cursor.execute("""
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL
)
""")
conexion.commit()
print("Tabla 'productos' creada exitosamente.")


### Operaciones CRUD (Create, Read, Update, Delete) directas en SQL


In [ ]:
# 2. CREATE: Insertar registros
productos_iniciales = [
    ("Licencia CRM", 120.0, 50),
    ("Módulo de Leads", 45.0, 100),
    ("API Gateway Pro", 250.0, 15)
]

cursor.executemany("INSERT INTO productos (nombre, precio, stock) VALUES (?, ?, ?)", productos_iniciales)
conexion.commit()
print(f"Registros insertados: {cursor.rowcount}")

# 3. READ: Consultar registros
cursor.execute("SELECT * FROM productos WHERE precio > 50.0")
filas = cursor.fetchall()
print("\nProductos con precio mayor a $50:")
for fila in filas:
    print(f"ID: {fila[0]} | Nombre: {fila[1]} | Precio: ${fila[2]} | Stock: {fila[3]}")

# 4. UPDATE: Actualizar stock de un producto
cursor.execute("UPDATE productos SET stock = stock - 1 WHERE nombre = ?", ("Licencia CRM",))
conexion.commit()

# 5. DELETE: Eliminar producto con stock cero o bajo (ejemplo simulado)
cursor.execute("DELETE FROM productos WHERE stock = 0")
conexion.commit()

# Cerramos el cursor y conexión al finalizar
cursor.close()
conexion.close()


## 2. Introducción al Mapeo Objeto-Relacional (ORM) con SQLAlchemy

Escribir consultas SQL en bruto como strings dentro de Python puede ser propenso a errores de tipado e inyección de código. Un **ORM (Object-Relational Mapping)** nos permite manipular la base de datos usando clases y objetos nativos de Python.

Utilizaremos **SQLAlchemy** (declarative style) para crear nuestros modelos.


In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

# Crear motor de base de datos en memoria para este ejemplo interactivo
engine = create_engine("sqlite:///:memory:", echo=False) # echo=True muestra el SQL generado
Base = declarative_base()

# Definición del Modelo (Mapea la clase a la tabla 'clientes')
class Cliente(Base):
    __tablename__ = 'clientes'
    
    id = Column(Integer, primary_key=True)
    nombre = Column(String(50), nullable=False)
    email = Column(String(50), unique=True, nullable=False)
    
    # Relación 1-a-N con Pedido
    pedidos = relationship("Pedido", back_populates="cliente", cascade="all, delete-orphan")

# Definición del Modelo de Pedidos
class Pedido(Base):
    __tablename__ = 'pedidos'
    
    id = Column(Integer, primary_key=True)
    monto = Column(Float, nullable=False)
    cliente_id = Column(Integer, ForeignKey('clientes.id'))
    
    cliente = relationship("Cliente", back_populates="pedidos")

# Crear las tablas físicamente en la base de datos
Base.metadata.create_all(engine)
print("Tablas 'clientes' y 'pedidos' generadas mediante SQLAlchemy.")


### Ejecución de operaciones mediante la Sesión de SQLAlchemy


In [ ]:
# Configurar la fábrica de sesiones
Session = sessionmaker(bind=engine)
sesion = Session()

# 1. CREATE: Insertar un cliente con un pedido
cliente_nuevo = Cliente(nombre="Sofía Rivas", email="sofia@umsa.edu.ar")
pedido_1 = Pedido(monto=350.0, cliente=cliente_nuevo)
pedido_2 = Pedido(monto=120.5, cliente=cliente_nuevo)

sesion.add(cliente_nuevo)
sesion.commit() # Guarda de forma atómica en la BD
print("Cliente y pedidos guardados con éxito.")

# 2. READ: Consultar registros a través de objetos de Python
cliente_db = sesion.query(Cliente).filter_by(email="sofia@umsa.edu.ar").first()
print(f"\nCliente encontrado: {cliente_db.nombre}")
print(f"Pedidos asociados a {cliente_db.nombre}:")
for ped in cliente_db.pedidos:
    print(f" -> Pedido ID: {ped.id} | Monto: ${ped.monto}")

# 3. UPDATE: Modificar el monto del pedido
pedido_db = sesion.query(Pedido).filter_by(id=1).first()
pedido_db.monto = 399.99
sesion.commit()
print(f"\nMonto del Pedido 1 modificado a: ${pedido_db.monto}")

# Cerramos la sesión
sesion.close()


---

## Desafío Práctico (Trabajo Práctico 4)

**Consigna de Negocio (Base de Datos de Suscripciones SaaS):**
Debes diseñar e implementar un backend relacional para una plataforma SaaS.

1. Define un modelo SQLAlchemy para `UsuarioSaaS` con campos:
   - `id` (entero, clave primaria)
   - `nombre` (string)
   - `email` (string, único)
2. Define un modelo `SuscripcionSaaS` con campos:
   - `id` (entero, clave primaria)
   - `plan` (string, ej. `"Basic"`, `"Enterprise"`)
   - `precio_mensual` (float)
   - `usuario_id` (entero, clave foránea vinculando a `UsuarioSaaS`)
3. Establece la relación de uno a muchos (o uno a uno, según lo consideres) indicando la vinculación entre ambos modelos.
4. Inicializa un motor SQLite en memoria y crea las tablas correspondientes.
5. Inicia una sesión e implementa las siguientes operaciones de prueba:
   - Inserta al menos 2 usuarios.
   - Asigna una suscripción de plan `"Enterprise"` ($199.99/mes) al usuario 1 y una suscripción de plan `"Basic"` ($29.99/mes) al usuario 2.
   - Ejecuta una consulta agregada que devuelva el ingreso mensual total de la plataforma (la suma de los precios mensuales de todas las suscripciones activas).
   - Modifica la suscripción del usuario 2 para actualizar su plan a `"Pro"` ($79.99/mes) y vuelve a consultar el ingreso total final para verificar la actualización de la persistencia.

Implementa tu solución a continuación.


In [ ]:
# Escribe la resolución aquí
# 1. Definir Modelos
# ...

# 2. Configurar Base de Datos e insertar registros
# ...

# 3. Consultas e ingresos totales
# ...
